# Iceberg Explorer — Enterprise Data Lakehouse

This notebook visualizes `polaris.analytics.transactions` through Trino and Iceberg metadata tables.

**Polaris catalog → Iceberg metadata → RustFS/S3 Parquet → Trino → Jupyter**.

In [ ]:
import os
import pandas as pd
import trino

conn = trino.dbapi.connect(
    host=os.environ['TRINO_HOST'], port=8080, user='data-analyst',
    catalog='polaris', schema='analytics'
)

def query(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [c[0] for c in cur.description]
    return pd.DataFrame(rows, columns=cols)


## 1. Business rows

In [ ]:
query('SELECT eventId, transactionId, amount, status, channel, country FROM transactions ORDER BY eventId')

## 2. Iceberg snapshots

In [ ]:
query('SELECT * FROM "transactions$snapshots" ORDER BY committed_at')

## 3. Iceberg history

In [ ]:
query('SELECT * FROM "transactions$history" ORDER BY made_current_at')

## 4. Physical Parquet files behind the table

In [ ]:
files = query('SELECT file_path, record_count, file_size_in_bytes FROM "transactions$files"')
files

## 5. File-size visualization

In [ ]:
ax = files.plot.bar(x='file_path', y='file_size_in_bytes', legend=False, title='Iceberg data files')
ax.set_ylabel('Bytes')

## Reading the architecture

- **RustFS/S3** stores physical Parquet and metadata files.
- **Iceberg** defines snapshots, manifests and transactional table semantics.
- **Polaris** catalogs the Iceberg table.
- **Trino** resolves the catalog and executes SQL.
- **Jupyter** is the analyst-facing exploration surface.